# <b>0. Preprocessing - BTC or ETH perp (Polars chunked).</b>

**Goal:** build the enriched maker-trade table for BTC perp.

**Key correctness rules (different from earlier versions):**
1. `hour` via `pl.from_epoch` - robust against any epoch shift, gives true UTC hour.
2. VPIN / RV / dist_to_mid use **strict half-open intervals** `[t - W, t)` - no self-leak.
3. Rolling features = **NaN** if window contains fewer than `N_MIN_OBS = 3` trades (early trades of first chunk + immediately after gaps).
4. Markout `pnl_{tau}s` = **NaN** if `t + tau > max_bbo_ts` (trades near the end of data).
5. Liquidation flow stays **0.0** when no liquidations happened - 0 is a valid value, not missing.
6. Subsampling is **chronologically stratified by global row index** - stable across chunks.

**Outputs:**
- `data/enriched/btc_full_parts/part_*.parquet` - full enriched data, one parquet per day.
- `data/enriched/btc_sub10_parts/part_*.parquet` - 10× subsample, one per day.
- `data/enriched/btc_sub10.parquet` - combined subsample for downstream notebooks.

**Resumable:** with `SKIP_EXISTING = True`, already-computed parts are skipped.

For first run, set `MAX_CHUNKS = 2` to test on two days, then `None` for full.

In [1]:
# imports.
import gc
import time
from pathlib import Path

import numpy as np
import polars as pl
from tqdm.auto import tqdm

from utilities_preprocess import (
    build_chunk_plan,
    compute_chunk,
    count_rows_parquet,
    make_subsample_for_chunk,
    safe_unlink,
    write_parquet,
    _collect_streaming,
    US_PER_SEC,
    N_MIN_OBS
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(50)

polars.config.Config

In [2]:
# paths — adjust DATA_ROOT if your layout differs.
DATA_ROOT       = Path("data")
RAW_TRADES      = DATA_ROOT / "binance_trades"       / "perp_ethusdt.parquet"
RAW_BBO         = DATA_ROOT / "binance_booktickers"  / "perp_ethusdt.parquet"
RAW_LIQ_BIN     = DATA_ROOT / "binance_liquidations" / "perp_ethusdt.parquet"
RAW_LIQ_BYB     = DATA_ROOT / "bybit_liquidations"   / "ethusdt.parquet"

ENRICHED_DIR    = DATA_ROOT / "enriched"
PARTS_DIR       = ENRICHED_DIR / "eth_full_parts"

# было sub10, стало sub40
SUB_PARTS_DIR   = ENRICHED_DIR / "eth_sub40_parts"
SUB_OUT         = ENRICHED_DIR / "eth_sub40.parquet"

ENRICHED_DIR.mkdir(parents=True, exist_ok=True)
PARTS_DIR.mkdir(parents=True, exist_ok=True)
SUB_PARTS_DIR.mkdir(parents=True, exist_ok=True)

## Constants and run settings

In [3]:
# fixed problem constants.
BYBIT_DELAY_US      = 200_000        # +200ms cross-exchange shift.
CLIPPED_NOTIONAL    = 100_000        # USD weight cap.
REBATE_BPS          = 0.5            # maker rebate in bps.

HORIZONS_S          = [30, 120, 300]
WINDOWS_S           = [5, 30, 120]

# BBO lookback for trade-time ASOF (10 min is plenty for crypto BBO density).
BBO_LOOKBACK_US     = 10 * 60 * US_PER_SEC

# chunking.
CHUNK_DAYS          = 1
ALIGN_TO_UTC_DAYS   = True

# resumability.
SKIP_EXISTING        = True
CLEAR_EXISTING_PARTS = False

# debug / test subset.
START_CHUNK_IDX     = 0
MAX_CHUNKS          = None            # set to None for full data after testing.

# subsample.
SUBSAMPLE_RATE      = 40

# output column order (final enriched schema, 25 cols).
OUTPUT_COLUMNS = [
    "timestamp",
    "price", "sign", "notional", "weight",
    "pnl_30s", "pnl_120s", "pnl_300s",
    "mid_price", "spread", "obi", "dist_to_mid",
    "hour",
    "vpin_5s", "vpin_30s", "vpin_120s",
    "rv_5s",   "rv_30s",   "rv_120s",
    "liq_binance_5s", "liq_binance_30s", "liq_binance_120s",
    "liq_bybit_5s",   "liq_bybit_30s",   "liq_bybit_120s",
]

print("CHUNK_DAYS:    ", CHUNK_DAYS)
print("MAX_CHUNKS:    ", MAX_CHUNKS)
print("SKIP_EXISTING: ", SKIP_EXISTING)
print("N_MIN_OBS:     ", N_MIN_OBS, "(from utilities_preprocess)")

CHUNK_DAYS:     1
MAX_CHUNKS:     None
SKIP_EXISTING:  True
N_MIN_OBS:      3 (from utilities_preprocess)


## <b>Build chunk plan.</b>

In [4]:
chunks, raw_ts_min, raw_ts_max = build_chunk_plan(
    RAW_TRADES,
    chunk_days=CHUNK_DAYS,
    align_to_utc=ALIGN_TO_UTC_DAYS,
)

chunks_to_run = list(enumerate(chunks))[START_CHUNK_IDX:]
if MAX_CHUNKS is not None:
    chunks_to_run = chunks_to_run[:MAX_CHUNKS]

print(f"raw_ts_min:        {raw_ts_min}")
print(f"raw_ts_max:        {raw_ts_max}")
print(f"total chunks:      {len(chunks)}")
print(f"chunks to run:     {len(chunks_to_run)}")

if chunks_to_run:
    i0, c0 = chunks_to_run[0]
    i1, c1 = chunks_to_run[-1]
    print(f"first run chunk:   idx={i0}, [{c0[0]}, {c0[1]})")
    print(f"last  run chunk:   idx={i1}, [{c1[0]}, {c1[1]})")

raw_ts_min:        1761955200058000
raw_ts_max:        1777420799990000
total chunks:      179
chunks to run:     179
first run chunk:   idx=0, [1761955200000000, 1762041600000000)
last  run chunk:   idx=178, [1777334400000000, 1777420799990001)


In [5]:
# optional cleanup. Keep False unless you intentionally want a full recomputation.
if CLEAR_EXISTING_PARTS:
    for d in [PARTS_DIR, SUB_PARTS_DIR]:
        for p in d.glob("part_*.parquet"):
            p.unlink()
    safe_unlink(SUB_OUT)
    print("Existing parts removed")
else:
    print("Existing parts are kept")

Existing parts are kept


## <b>Execute chunked preprocessing.<b>

Outer tqdm shows chunk progress. Per chunk reports rows, output size, and elapsed time.

In [6]:
chunk_log = []
run_t0 = time.time()

global_row_offset = 0

# recover offset from existing previous parts (if resuming).
if START_CHUNK_IDX > 0:
    print("Recovering global row offset from previous parts...")
    for prev_idx in tqdm(range(START_CHUNK_IDX), desc="counting previous parts"):
        prev_part = PARTS_DIR / f"part_{prev_idx:05d}.parquet"
        assert prev_part.exists(), f"Cannot recover offset: missing {prev_part}"
        global_row_offset += count_rows_parquet(prev_part)
    print("global_row_offset:", global_row_offset)

pbar = tqdm(chunks_to_run, total=len(chunks_to_run), desc="BTC chunks", unit="chunk")

for chunk_idx, (chunk_start_us, chunk_end_us) in pbar:
    part_path     = PARTS_DIR    / f"part_{chunk_idx:05d}.parquet"
    sub_part_path = SUB_PARTS_DIR / f"part_{chunk_idx:05d}.parquet"

    part_exists = part_path.exists() and part_path.stat().st_size > 0
    sub_exists  = sub_part_path.exists() and sub_part_path.stat().st_size > 0

    # skip already computed.
    if SKIP_EXISTING and part_exists:
        n_rows = count_rows_parquet(part_path)

        # rebuild subsample part if missing.
        if not sub_exists:
            part_df = pl.read_parquet(str(part_path))
            sub_df = make_subsample_for_chunk(part_df, global_row_offset, SUBSAMPLE_RATE)
            write_parquet(sub_df, sub_part_path)
            del part_df, sub_df
            gc.collect()

        global_row_offset += n_rows
        pbar.set_postfix_str(f"skip part_{chunk_idx:05d}, rows={n_rows:,}, offset={global_row_offset:,}")
        continue

    safe_unlink(part_path)
    safe_unlink(sub_part_path)

    t0 = time.time()

    df = compute_chunk(
        chunk_start_us, chunk_end_us,
        raw_trades_path  = RAW_TRADES,
        raw_bbo_path     = RAW_BBO,
        raw_liq_bin_path = RAW_LIQ_BIN,
        raw_liq_byb_path = RAW_LIQ_BYB,
        horizons_s       = HORIZONS_S,
        windows_s        = WINDOWS_S,
        bbo_lookback_us  = BBO_LOOKBACK_US,
        bybit_delay_us   = BYBIT_DELAY_US,
        clipped_notional = CLIPPED_NOTIONAL,
        rebate_bps       = REBATE_BPS,
        output_columns   = OUTPUT_COLUMNS,
        n_min_obs        = N_MIN_OBS,
    )
    n_rows = df.height

    sub_df = make_subsample_for_chunk(df, global_row_offset, SUBSAMPLE_RATE)

    write_parquet(df, part_path)
    write_parquet(sub_df, sub_part_path)

    elapsed = time.time() - t0
    global_row_offset += n_rows

    info = {
        "chunk_idx":        chunk_idx,
        "rows":             n_rows,
        "sub_rows":         sub_df.height,
        "full_gb":          part_path.stat().st_size / 1024**3,
        "sub_gb":           sub_part_path.stat().st_size / 1024**3,
        "time_min":         elapsed / 60,
        "global_offset":    global_row_offset,
    }
    chunk_log.append(info)

    print(
        f"part_{chunk_idx:05d}: "
        f"rows={n_rows:,}, "
        f"full={info['full_gb']:.3f} GB, "
        f"sub={info['sub_gb']:.3f} GB, "
        f"time={info['time_min']:.2f} min, "
        f"offset={global_row_offset:,}"
    )
    pbar.set_postfix_str(f"last_min={info['time_min']:.2f}")

    del df, sub_df
    gc.collect()

run_elapsed = time.time() - run_t0
print(f"\nChunk loop finished in {run_elapsed / 60:.2f} min")
print(f"Computed new chunks: {len(chunk_log)}")
print(f"Global row offset:   {global_row_offset:,}")

BTC chunks:   0%|          | 0/179 [00:00<?, ?chunk/s]

part_00000: rows=2,478,182, full=0.038 GB, sub=0.003 GB, time=0.09 min, offset=2,478,182
part_00001: rows=3,238,845, full=0.049 GB, sub=0.004 GB, time=0.11 min, offset=5,717,027
part_00002: rows=10,070,803, full=0.158 GB, sub=0.011 GB, time=0.37 min, offset=15,787,830
part_00003: rows=15,897,417, full=0.254 GB, sub=0.018 GB, time=0.63 min, offset=31,685,247
part_00004: rows=10,503,612, full=0.158 GB, sub=0.012 GB, time=0.49 min, offset=42,188,859
part_00005: rows=8,995,145, full=0.134 GB, sub=0.010 GB, time=0.51 min, offset=51,184,004
part_00006: rows=10,160,525, full=0.153 GB, sub=0.012 GB, time=0.58 min, offset=61,344,529
part_00007: rows=5,441,778, full=0.080 GB, sub=0.006 GB, time=0.21 min, offset=66,786,307
part_00008: rows=6,039,399, full=0.090 GB, sub=0.007 GB, time=0.25 min, offset=72,825,706
part_00009: rows=7,309,288, full=0.107 GB, sub=0.008 GB, time=0.28 min, offset=80,134,994
part_00010: rows=8,880,427, full=0.128 GB, sub=0.010 GB, time=0.32 min, offset=89,015,421
part_000

## <b>Verify parts: row count check.</b>

In [7]:
parts     = sorted(PARTS_DIR.glob("part_*.parquet"))
sub_parts = sorted(SUB_PARTS_DIR.glob("part_*.parquet"))

print(f"Full parts: {len(parts)} / {len(chunks)}")
print(f"Sub parts:  {len(sub_parts)} / {len(chunks)}")

assert parts,     "No full parts found"
assert sub_parts, "No sub parts found"

# total rows in parts vs raw trades (only valid when all chunks done).
parts_count = int(
    _collect_streaming(pl.scan_parquet(str(PARTS_DIR / "*.parquet")).select(pl.len().alias("n")))[0, "n"]
)
raw_count_total = int(
    _collect_streaming(pl.scan_parquet(str(RAW_TRADES)).select(pl.len().alias("n")))[0, "n"]
)

print(f"Raw trades total: {raw_count_total:>15,}")
print(f"Parts rows:       {parts_count:>15,}")

if len(parts) == len(chunks):
    assert parts_count == raw_count_total, f"Row count mismatch: {parts_count} != {raw_count_total}"
    print("Full row-count check passed")
else:
    print("Partial run: row-count assertion skipped (will fire on full run)")

Full parts: 179 / 179
Sub parts:  179 / 179
Raw trades total:   1,370,451,261
Parts rows:         1,370,451,261
Full row-count check passed


## <b>Sanity checks on enriched parts.</b>

We verify value ranges and NaN counts. Expected NaN rows:
- **markout** `pnl_{tau}s`: trades at the very end of data (t + tau > last BBO).
- **rolling** `vpin_W`, `rv_W`: early trades whose `[t - W, t)` window contains fewer
  than `N_MIN_OBS = 3` trades. This is normal for the first trades of the dataset
  and after very long gaps.
- **liquidation flow** is **NOT** NaN - 0 is a valid value.

In [8]:
lf_full = pl.scan_parquet(str(PARTS_DIR / "*.parquet"))

summary = _collect_streaming(
    lf_full.select([
        pl.len().alias("n"),
        # markout NaN counts (NaN, not null — Polars treats them separately).
        pl.col("pnl_30s").is_nan().sum().alias("nan_pnl_30s"),
        pl.col("pnl_120s").is_nan().sum().alias("nan_pnl_120s"),
        pl.col("pnl_300s").is_nan().sum().alias("nan_pnl_300s"),
        # rolling NaN counts.
        pl.col("vpin_5s").is_nan().sum().alias("nan_vpin_5s"),
        pl.col("vpin_30s").is_nan().sum().alias("nan_vpin_30s"),
        pl.col("vpin_120s").is_nan().sum().alias("nan_vpin_120s"),
        pl.col("rv_30s").is_nan().sum().alias("nan_rv_30s"),
        # PnL means (NaN-aware).
        pl.col("pnl_30s").filter(pl.col("pnl_30s").is_not_nan()).mean().alias("mean_pnl_30s"),
        pl.col("pnl_120s").filter(pl.col("pnl_120s").is_not_nan()).mean().alias("mean_pnl_120s"),
        pl.col("pnl_300s").filter(pl.col("pnl_300s").is_not_nan()).mean().alias("mean_pnl_300s"),
        # ranges.
        pl.col("weight").max().alias("max_weight"),
        pl.col("sign").min().alias("min_sign"),
        pl.col("sign").max().alias("max_sign"),
        pl.col("hour").min().alias("min_hour"),
        pl.col("hour").max().alias("max_hour"),
        pl.col("obi").min().alias("min_obi"),
        pl.col("obi").max().alias("max_obi"),
        pl.col("vpin_30s").min().alias("min_vpin_30s"),
        pl.col("vpin_30s").max().alias("max_vpin_30s"),
        pl.col("liq_binance_30s").abs().mean().alias("avg_abs_liq_binance"),
        pl.col("liq_bybit_30s").abs().mean().alias("avg_abs_liq_bybit"),
        pl.col("timestamp").min().alias("ts_min"),
        pl.col("timestamp").max().alias("ts_max"),
    ])
)
print(summary.transpose(include_header=True, header_name="metric"))

row = summary.row(0, named=True)
n = row["n"]
print()

# hard asserts.
assert row["max_weight"] <= CLIPPED_NOTIONAL + 1e-6, "weight not capped"
assert row["min_sign"] == -1 and row["max_sign"] == 1, "sign must be +/-1"
assert 0 <= row["min_hour"] <= 23 and 0 <= row["max_hour"] <= 23, "hour out of [0,23]"
assert row["min_obi"]  is None or row["min_obi"]  >= 0, "obi < 0"
assert row["max_obi"]  is None or row["max_obi"]  <= 1, "obi > 1"
assert row["min_vpin_30s"] is None or row["min_vpin_30s"] >= 0, "vpin < 0"
assert row["max_vpin_30s"] is None or row["max_vpin_30s"] <= 1.001, "vpin > 1"

# NaN counts — must be a small fraction.
def pct(x): return 100.0 * x / n
print(f"NaN pnl_30s:   {row['nan_pnl_30s']:>10,}  ({pct(row['nan_pnl_30s']):.4f}%)  -- expected: ~0 except final trades")
print(f"NaN pnl_300s:  {row['nan_pnl_300s']:>10,}  ({pct(row['nan_pnl_300s']):.4f}%)  -- expected: trades near data end")
print(f"NaN vpin_5s:   {row['nan_vpin_5s']:>10,}  ({pct(row['nan_vpin_5s']):.4f}%)  -- expected: early/sparse-window trades")
print(f"NaN vpin_120s: {row['nan_vpin_120s']:>10,}  ({pct(row['nan_vpin_120s']):.4f}%)")
print(f"NaN rv_30s:    {row['nan_rv_30s']:>10,}  ({pct(row['nan_rv_30s']):.4f}%)")

assert pct(row['nan_pnl_30s'])   < 0.5, "too many NaN pnl_30s"
assert pct(row['nan_vpin_120s']) < 1.0, "too many NaN vpin_120s"
assert pct(row['nan_rv_30s'])    < 1.0, "too many NaN rv_30s"

# mean PnL plausibility.
for col in ["mean_pnl_30s", "mean_pnl_120s", "mean_pnl_300s"]:
    v = row[col]
    assert v is not None and -2 < v < 3, f"{col} = {v} out of plausible range"

print("\nAll sanity checks passed")

shape: (24, 2)
┌─────────────────────┬────────────┐
│ metric              ┆ column_0   │
│ ---                 ┆ ---        │
│ str                 ┆ f64        │
╞═════════════════════╪════════════╡
│ n                   ┆ 1.3705e9   │
│ nan_pnl_30s         ┆ 242.0      │
│ nan_pnl_120s        ┆ 1729.0     │
│ nan_pnl_300s        ┆ 5583.0     │
│ nan_vpin_5s         ┆ 28525.0    │
│ nan_vpin_30s        ┆ 39.0       │
│ nan_vpin_120s       ┆ 28.0       │
│ nan_rv_30s          ┆ 39.0       │
│ mean_pnl_30s        ┆ -0.162977  │
│ mean_pnl_120s       ┆ -0.141281  │
│ …                   ┆ …          │
│ min_hour            ┆ 0.0        │
│ max_hour            ┆ 23.0       │
│ min_obi             ┆ 1.9820e-7  │
│ max_obi             ┆ 1.0        │
│ min_vpin_30s        ┆ 6.4242e-10 │
│ max_vpin_30s        ┆ 1.0        │
│ avg_abs_liq_binance ┆ 10.626558  │
│ avg_abs_liq_bybit   ┆ 21.703287  │
│ ts_min              ┆ 1.7620e15  │
│ ts_max              ┆ 1.7774e15  │
└─────────────────────┴

## <b>Combine `{instrument}_sub40.parquet`.</b>

In [9]:
safe_unlink(SUB_OUT)

print("Combining sub10 parts...")
t0 = time.time()

lf_sub = pl.concat([
    pl.scan_parquet(str(p))
    for p in sorted(SUB_PARTS_DIR.glob("part_*.parquet"))
])

try:
    lf_sub.sink_parquet(str(SUB_OUT), compression="zstd", statistics=True)
except Exception as e:
    print("sink_parquet failed, fallback to collect().write_parquet()")
    print("Reason:", repr(e))
    df_sub = _collect_streaming(lf_sub)
    write_parquet(df_sub, SUB_OUT)
    del df_sub
    gc.collect()

elapsed = time.time() - t0
sub_n = int(_collect_streaming(pl.scan_parquet(str(SUB_OUT)).select(pl.len().alias("n")))[0, "n"])

print(f"Done in {elapsed / 60:.2f} min")
print(f"Sub10 rows: {sub_n:,}")
print(f"Sub10 size: {SUB_OUT.stat().st_size / 1024**3:.2f} GB")

Combining sub10 parts...
Done in 0.15 min
Sub10 rows: 34,261,282
Sub10 size: 1.51 GB


## <b>Cross-check: subsample preserves aggregate stats.</b>

Heavy-tailed PnL -> relative diff is meaningless. We compare absolute diff to standard error of the subsample mean. Difference should be < 5 * SE_sub.

In [10]:
def get_stats_lf(lf: pl.LazyFrame) -> dict:
    # base stats: filter where ALL three pnl are not NaN.
    out_struct = _collect_streaming(
        lf.select([
            pl.col("weight").mean().alias("avg_weight"),
            pl.col("vpin_30s").filter(pl.col("vpin_30s").is_not_nan()).mean().alias("avg_vpin_30s"),
            pl.col("spread").mean().alias("avg_spread"),
        ])
    ).row(0, named=True)

    # PnL stats: each horizon filtered independently.
    results = dict(out_struct)
    for tau in [30, 120, 300]:
        col = f"pnl_{tau}s"
        sub = _collect_streaming(
            lf.filter(pl.col(col).is_not_nan())
              .select([
                  ((pl.col("weight") * pl.col(col)).sum() / pl.col("weight").sum()).alias(f"pnl_all_{tau}s"),
                  pl.col(col).std().alias(f"std_pnl_{tau}s"),
                  pl.len().alias(f"n_valid_{tau}s"),
              ])
        ).row(0, named=True)
        results.update(sub)
    return results

full_stats = get_stats_lf(pl.scan_parquet(str(PARTS_DIR / "*.parquet")))
sub_stats  = get_stats_lf(pl.scan_parquet(str(SUB_OUT)))

# PnL columns: assert abs_diff < 5 * SE_sub.
print(f"{'metric':<20} {'full':>15} {'sub':>15} {'abs_diff':>10} {'5·SE_sub':>10}")
print("-" * 75)
for tau in [30, 120, 300]:
    col = f"pnl_all_{tau}s"
    std_col = f"std_pnl_{tau}s"
    n_col = f"n_valid_{tau}s"
    f_val = full_stats[col]
    s_val = sub_stats[col]
    abs_d = abs(s_val - f_val)
    se_sub = sub_stats[std_col] / np.sqrt(sub_stats[n_col])
    print(f"{col:<20} {f_val:>15.6f} {s_val:>15.6f} {abs_d:>10.6f} {5*se_sub:>10.6f}")
    assert abs_d < 5 * se_sub, f"{col}: abs_diff {abs_d:.6f} > 5·SE {5*se_sub:.6f}"

# Structural columns: relative diff < 1%.
print()
for col in ["avg_weight", "avg_vpin_30s", "avg_spread"]:
    f_val = full_stats[col]
    s_val = sub_stats[col]
    rel = abs(s_val - f_val) / abs(f_val) * 100 if f_val and abs(f_val) > 1e-9 else 0
    print(f"{col:<20} {f_val:>15.6f} {s_val:>15.6f} {rel:>9.3f}%")
    assert rel < 1.0, f"{col}: relative diff {rel:.3f}% > 1%"

print("\nSubsample preserves aggregate statistics")

metric                          full             sub   abs_diff   5·SE_sub
---------------------------------------------------------------------------
pnl_all_30s                -0.048946       -0.055310   0.006364   0.011517
pnl_all_120s                0.008628        0.000103   0.008526   0.020457
pnl_all_300s                0.063414        0.043512   0.019902   0.029999

avg_weight               1483.039062     1484.756592     0.116%
avg_vpin_30s                0.303494        0.303490     0.001%
avg_spread                  0.019665        0.019667     0.007%

Subsample preserves aggregate statistics


## <b>Preview enriched data.</b>

In [11]:
preview = pl.read_parquet(str(SUB_OUT), n_rows=5)
print("Sub10 preview (first 5 rows):")
print(preview)

print("\nDtypes:")
for name, dtype in zip(preview.columns, preview.dtypes):
    print(f"  {name:<22} {dtype}")

Sub10 preview (first 5 rows):
shape: (5, 25)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ tim ┆ pri ┆ sig ┆ not ┆ wei ┆ pnl ┆ pnl ┆ pnl ┆ mid ┆ spr ┆ obi ┆ dis ┆ hou ┆ vpi ┆ vpi ┆ vpi ┆ rv_ ┆ rv_ ┆ rv_ ┆ liq ┆ liq ┆ liq ┆ liq ┆ liq ┆ liq │
│ est ┆ ce  ┆ n   ┆ ion ┆ ght ┆ _30 ┆ _12 ┆ _30 ┆ _pr ┆ ead ┆ --- ┆ t_t ┆ r   ┆ n_5 ┆ n_3 ┆ n_1 ┆ 5s  ┆ 30s ┆ 120 ┆ _bi ┆ _bi ┆ _bi ┆ _by ┆ _by ┆ _by │
│ amp ┆ --- ┆ --- ┆ al  ┆ --- ┆ s   ┆ 0s  ┆ 0s  ┆ ice ┆ --- ┆ f32 ┆ o_m ┆ --- ┆ s   ┆ 0s  ┆ 20s ┆ --- ┆ --- ┆ s   ┆ nan ┆ nan ┆ nan ┆ bit ┆ bit ┆ bit │
│ --- ┆ f64 ┆ i8  ┆ --- ┆ f32 ┆ --- ┆ --- ┆ --- ┆ --- ┆ f32 ┆     ┆ id  ┆ i8  ┆ --- ┆ --- ┆ --- ┆ f32 ┆ f32 ┆ --- ┆ ce_ ┆ ce_ ┆ ce_ ┆ _5s ┆ _30 ┆ _12 │
│ i64 ┆     ┆     ┆ f32 ┆     ┆ f32 ┆ f32 ┆ f32 ┆ f32 ┆     ┆     ┆ --- ┆     ┆ f32 ┆ f32 ┆ f32 ┆     ┆     ┆ f32 ┆ 5s  ┆ 30s ┆ 120 ┆ --- ┆ s   ┆ 0s  │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆

In [12]:
print("Final artifacts:")
print(f"  full parts: {PARTS_DIR}/part_*.parquet")
print(f"  sub parts:  {SUB_PARTS_DIR}/part_*.parquet")
print(f"  sub10:      {SUB_OUT}  ({SUB_OUT.stat().st_size / 1024**3:.2f} GB)")
print("\nDownstream notebooks read btc_sub10.parquet (lives comfortably in RAM).")

Final artifacts:
  full parts: data\enriched\eth_full_parts/part_*.parquet
  sub parts:  data\enriched\eth_sub40_parts/part_*.parquet
  sub10:      data\enriched\eth_sub40.parquet  (1.51 GB)

Downstream notebooks read btc_sub10.parquet (lives comfortably in RAM).
